In [1]:
import math
import numpy as np

In [2]:
# --- Global Parameters ---
p_warp_size = 32

# --- Global Functions ---
def f_print_title(title_name):
    print(f"*----------------------------*")
    print(f"{title_name}:")
    print(f"*----------------------------*")

# Returns occupancy level of each SM scenario as a percentage (%).
# Per slide 5 - GPU HW Arch 201: memory hierarchy. Parallel performance theory. GEMM kernels. 
def f_occupancy_level (active_warps_per_sm, max_warps_per_sm):
    occupancy_level = (active_warps_per_sm / max_warps_per_sm)
    return (occupancy_level*100)

# integer return of the total number of threads per SM based on threads per block and blocks per SM
def f_threads_per_sm(threads_per_block, blocks_per_sm):
    total_threads_per_sm = (threads_per_block * blocks_per_sm)
    return total_threads_per_sm

# boolean check that is TRUE when total threads per SM is below maximum threads per SM  [0,{max_threads_per_sm}]
def f_valid_threads_per_sm(total_threads_per_sm, max_threads_per_sm):
    valid_threads_per_sm = (total_threads_per_sm <= max_threads_per_sm)
    return valid_threads_per_sm

# integer return of the total registers per SM based on total threads per SM and registers per thread
def f_registers_per_sm(total_threads_per_sm, registers_per_thread):
    total_registers_per_sm = (total_threads_per_sm * registers_per_thread)
    return total_registers_per_sm

# boolean check that is TRUE when total registers per SM is below maximum registers per SM  [0,{max_registers_per_sm}]
def f_valid_registers_per_sm(total_registers_per_sm, max_registers_per_sm):
    valid_registers_per_sm = (total_registers_per_sm <= max_registers_per_sm)
    return valid_registers_per_sm

![Question 1](images/q1.png)

In [3]:
# Following block of code will go through each X-thread block iteration and then multiply it 
# by the number of threads. If the thread count goes beyond the physical limit of the 
# CUDA SM (1536 threads) then discard it. Whatever value is closest to 1536 threads wins.

# # Problem Parameters
threads = [128, 256, 512, 1024] # threads per block
blocks = [1, 2, 3 , 4] # maximum blocks per SM
solution_space = np.zeros((len(blocks), len(threads)),dtype=int)

max_threads_per_sm = 1536 # maximum threads per SM
optimal_threads_per_sm = 0 # winner winner chicken dinner solution
optimal_blocks = None
optimal_threads = None

# Problem Scoreboard
def f_print_matrix_sm(solution, blocks, threads): 
    print(f"Threads per SM: {solution}")
    print(f"Blocks: {blocks}")
    print(f"Threads per Block: {threads}")

# --- iterate through block*thread perturbations to explore SM solution space ---
# iterate through different block counts per SM
for i in range (len(blocks)):
    # iterate through different thread counts per block
    for j in range (len(threads)):
        solution_space[i,j] = blocks[i] * threads[j]

        # --- Find the best fit solution ---
        if (solution_space[i,j] <= max_threads_per_sm):
            if (optimal_threads_per_sm < solution_space[i,j]):
                optimal_threads_per_sm = solution_space[i,j]
                optimal_blocks = blocks[i]
                optimal_threads = threads[j]

print(f"--- Best Solution ---")
f_print_matrix_sm(optimal_threads_per_sm, optimal_blocks, optimal_threads)

--- Best Solution ---
Threads per SM: 1536
Blocks: 3
Threads per Block: 512


![Question 2](images/q2.png)

In [4]:
threads = 512 # threads per block
vector_length = 2000

blocks = math.ceil(vector_length/threads)
threads_per_grid = threads * blocks
print(f"Threads per Grid: {threads_per_grid}")

Threads per Grid: 2048


![Question 3](images/q3.png)

In [5]:
warp_size = 32
leftovers = threads_per_grid - vector_length
diverged_warps = math.floor(leftovers / warp_size)

# Threads 2000 % 32 = 16 -> meaning there are 16 threads out of 32 that will exist under the 
# 2000 vector length limit. There will 16 threads above the 2000 vector length limit that fail.
# Threads 2017-2048 will all fail the vector length limit together, so the warp doesn't diverge.
print(f"Diverged Warps: {diverged_warps}")

Diverged Warps: 1


![Question 4](images/q4.png)

In [6]:
# --- Parameters ---
p_max_threads_per_sm = 2048 # parameter: maximum threads per SM
p_max_warps_per_sm = p_max_threads_per_sm / p_warp_size
p_max_blocks_per_sm = 64
blocks = [8, 16, 32, 64, 32]
threads = [128, 64, 32, 32, 64]

# --- Variables ---
# SM size variables
threads_per_sm = 0 # matrix of maximum number of threads per SM
valid_threads_per_sm = 0 # matrix of valid threads per SM

# Occupancy level variables
active_warps_per_sm = 0
occupancy_level_per_sm = 0 # matrix of occupancy levels per SM

# --- Functions ---
#  Determine valid SMs and print occupancy levels
for i in range(len(blocks)):
    # Each block has an associated thread count, no need to do a perturbation.
    threads_per_sm = blocks[i] * threads[i]
    # valid SM boolean
    valid_threads_per_sm = (threads_per_sm <= p_max_threads_per_sm
                            and blocks[i] <= p_max_blocks_per_sm)

    #  Determine if you have a valid SM, and if so, print associated occupancy level 
    if(valid_threads_per_sm):
        active_warps_per_sm = math.ceil(threads_per_sm / p_warp_size)
        occupancy_level_per_sm = f_occupancy_level(active_warps_per_sm, p_max_warps_per_sm)

        f_print_title("Valid SM:")
        print(f"Threads per SM: {threads_per_sm}")
        print(f"Blocks per SM: {blocks[i]}")
        print(f"Threads per Block: {threads[i]}")
        print(f"SM Occupancy Level: {occupancy_level_per_sm}%")

*----------------------------*
Valid SM::
*----------------------------*
Threads per SM: 1024
Blocks per SM: 8
Threads per Block: 128
SM Occupancy Level: 50.0%
*----------------------------*
Valid SM::
*----------------------------*
Threads per SM: 1024
Blocks per SM: 16
Threads per Block: 64
SM Occupancy Level: 50.0%
*----------------------------*
Valid SM::
*----------------------------*
Threads per SM: 1024
Blocks per SM: 32
Threads per Block: 32
SM Occupancy Level: 50.0%
*----------------------------*
Valid SM::
*----------------------------*
Threads per SM: 2048
Blocks per SM: 64
Threads per Block: 32
SM Occupancy Level: 100.0%
*----------------------------*
Valid SM::
*----------------------------*
Threads per SM: 2048
Blocks per SM: 32
Threads per Block: 64
SM Occupancy Level: 100.0%


![Question 5](images/q5.png)

In [7]:
# --- Parameters ---
# problem parameters
p_max_threads_per_sm = 2048 # parameter: maximum threads per SM
p_max_warps_per_sm = (p_max_threads_per_sm / p_warp_size)
p_max_blocks_per_sm = 32
p_threads_per_warp = 32
p_max_registers_per_sm = (2**16)
# problem scenarios
threads = [128, 32, 256]
registers_per_thread = [30, 29, 34]

# --- Variables ---
# SM size variables
thread_blocks_per_sm = 0
register_blocks_per_sm = 0

# Occupancy level variables
active_warps_per_sm = 0
occupancy_level_per_sm = 0 # matrix of occupancy levels per SM

# --- Functions ---
# 1. Find the max number of blocks that can fit in each SM
for i in range(len(threads)):
    # Determine the maximum supported blocks in the SM per scenario
    registers_per_block = threads[i] * registers_per_thread[i]

    thread_blocks_per_sm = p_max_threads_per_sm / threads[i]
    register_blocks_per_sm = math.floor(p_max_registers_per_sm / registers_per_block)
    # Determine which factor is the limiting input
    active_blocks = min(thread_blocks_per_sm, register_blocks_per_sm, p_max_blocks_per_sm)

    # Occupancy math
    active_warps_per_sm = (active_blocks * threads[i]) / p_threads_per_warp
    occupancy_level_per_sm = (active_warps_per_sm / p_max_warps_per_sm) * 100

    # terminal information 
    f_print_title(f"Scenario {i}")
    print(f"Thread Blocks per SM: {thread_blocks_per_sm}")
    print(f"Register Blocks per SM: {register_blocks_per_sm}")
    print(f"Max Blocks per SM: {p_max_blocks_per_sm}")
    print(f"Active Blocks (upper bound defined by lowest block count): {active_blocks}")
    print(f"Active Warps: {active_warps_per_sm}")
    print(f"Occupancy Level: {occupancy_level_per_sm}%")


*----------------------------*
Scenario 0:
*----------------------------*
Thread Blocks per SM: 16.0
Register Blocks per SM: 17
Max Blocks per SM: 32
Active Blocks (upper bound defined by lowest block count): 16.0
Active Warps: 64.0
Occupancy Level: 100.0%
*----------------------------*
Scenario 1:
*----------------------------*
Thread Blocks per SM: 64.0
Register Blocks per SM: 70
Max Blocks per SM: 32
Active Blocks (upper bound defined by lowest block count): 32
Active Warps: 32.0
Occupancy Level: 50.0%
*----------------------------*
Scenario 2:
*----------------------------*
Thread Blocks per SM: 8.0
Register Blocks per SM: 7
Max Blocks per SM: 32
Active Blocks (upper bound defined by lowest block count): 7
Active Warps: 56.0
Occupancy Level: 87.5%


For scenario 0 is able to achieve 100% occupancy. 

For scenario 1 only 50% occupancy can be achieved, limited by the SM block count parameter of 32.

For scenario 2 only 87.5% occupancy can be achieved, limited by the number of available registers in the SM.

![Control Divergence Analysis](images/analysis_control_divergence.png)

### Assumptions
Referring to lecture 4, the MMULT Kernel assumptions are:
1. 16x16 tiles and thread blocks
2. Each thread block has 8 warps ((16 threads x 16 threads) / (1 warp) = 256 threads / 32 threads = 8 warps)
3. Square 100x100 matrix
4. Each thread will go through 7 phases (ceiling of 100/16)
5. 49 thread blocks / tiles

### Type 1
For an `N` matrix there are 6 full block columns (`M` does block rows) across 7 rows. Thus 6 * 7 = 42 Type 1 blocks.

At 8 warps per block, there's a total of 42 * 8 = 336 warps. So far, similar to the `M` matrix. Each warp has 7 phases (336 warps * 7 phases) = 2352 Type 1 warp-phases. 

For the first six block rows, there are zero divergent warp-phases. Take threads 0-15 and 16-31 in each block for example, the threads will all process valid data. For block 7, rows 96-97 (warp 1 in block 7) will cover one warp, then rows 98-99 (warp 2 in block 7) will cover the last warp for the 100x100 MMULT matrix. Rows 100-111 will all process valid zero data. There's no divergence since all the warps in block 7 process valid data.

Type 1 Diverged Warps: 0
Type 1 Performance Impact: 0 / 2352 = 0% performance impact

### Type 2
For the `N` matrix, the 7th column is where divergence occurs. There a total of (8 warps per block * 7 blocks * 7 phases) = 392 total warp-phases for column 7.

All warps in block rows 1-6 experience divergence between threads in column 0-3 versus 4-15, which causes each warp to diverge. Then for block row 7, only the two warps along rows 96-99 diverge, meanwhile warps 3-8 all process valid zero data.

For blocks 1-6, (8 warps per block x 6 phases) = 48 divergent warp-phases, but I need to account for the two divergences in block row 7 for each phase. Thus 48+2 = 50 divergent warp-phases. 

Multiply the result across the 7 rows, (50 divergent warp phases * 7 block rows) = 350 total divergent warps-phases in `N`.

Type 1 Diverged Warps: 350
Type 1 Performance Impact: 350 / 392 = 89.29% performance impact

**Combined Total** = divergent warp-phases / total warp-phases = (350 + 0) / (2352 + 392) = 350 / 2744 = 12.76%

### Q: When will the warp divergence impact become minimal?
When the `M` or `N` matrix widths are base-2 number, then there should be no warp divergence for the MMUlT algorithm.

#### Q: How does varying tile size affect arithmetic intensity
Based on this assignment, a larger tile means that more phases can be computed for the same thread-block. Thus I would say increasing tile size increases arithmetic intensity since more operations are performed with only one memory traffic byte for example, versus a smaller tile size would mean fewer phases and thus fewer operations per memory traffic byte.